# RSNA Knee EDA: duplicate reports, a broken flag, and the real language mix

Twelve binary findings per knee-MRI study — `ACL`, `MCL`, the two menisci, three
osteoarthritis compartments, `Effusion`, `Synovitis`, `Baker's`, `Contusion`,
`Fracture` — scored by macro-averaged AUC ROC.

Several good EDAs already cover the ground you would expect: label prevalence, the
co-occurrence matrix, plane and sequence counts, report length, sample DICOM slices.
**This notebook deliberately does not repeat those plots.** It looks at the tabular
files for the things that change what you build, and finds four:

1. **`Fluid_Sensitive` and `Fat_Suppression` are byte-identical** across all 24,371
   series. They describe different physics. One of them is not real data.
2. **The quick substring trick for language ID is badly broken** on this corpus — the
   version in circulation labels *zero* reports as Spanish while calling 422 Spanish
   and 416 Turkish reports French. The real mix is seven Latin-script languages plus
   Greek and Cyrillic.
3. **Report text is not unique.** 46 report texts are shared by 177 studies, mostly
   normal-study boilerplate — one identical template covers 37 studies. A random
   split puts the same text in train and validation.
4. **`Report` exists only in `train.csv`.** The test schema has no report column, so
   the text is a training-time labelling signal, not a model input.

Tabular files only, no DICOM read, a few seconds on CPU.

In [1]:
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

KAGGLE_INPUT = Path("/kaggle/input")
COMP = "rsna-knee-abnormality-detection"


def find_data_dir():
    """Locate the tabular files, on Kaggle or in a local checkout."""
    for p in [KAGGLE_INPUT / "competitions" / COMP, KAGGLE_INPUT / COMP,
              Path("../data"), Path("data")]:
        if (p / "train.csv").exists():
            return p
    # Kaggle mounts competition data in more than one layout. Search by the file
    # we need, bounded to two levels so it never walks into the DICOM tree.
    for pattern in ("*/train.csv", "*/*/train.csv"):
        hits = sorted(KAGGLE_INPUT.glob(pattern))
        if hits:
            return hits[0].parent
    return None


DATA = find_data_dir()
if DATA is None:
    mounted = sorted(str(p) for p in KAGGLE_INPUT.glob("*")) + \
              sorted(str(p) for p in KAGGLE_INPUT.glob("*/*"))
    raise FileNotFoundError(f"train.csv not found. /kaggle/input holds: {mounted or 'nothing'}")
print(f"data directory: {DATA}")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

train = pd.read_csv(DATA / "train.csv")
series = pd.read_csv(DATA / "train_series.csv")
test = pd.read_csv(DATA / "test.csv")
test_series = pd.read_csv(DATA / "test_series.csv")
sample_sub = pd.read_csv(DATA / "sample_submission.csv")

LABELS = [c for c in sample_sub.columns if c != "StudyInstanceUID"]
print(f"{len(LABELS)} targets: {', '.join(LABELS)}")

In [2]:
# One visual language for the whole notebook. Surfaces are pinned light because
# matplotlib output is a static PNG — it must stay legible under a dark UI theme.
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
BLUE, ORANGE, MUTED = "#2a78d6", "#eb6834", "#a9a8a2"

plt.rcParams.update({
    "figure.dpi": 130,
    "figure.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID,
    "axes.linewidth": 0.8,
    "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK,
    "axes.titlesize": 11,
    "axes.titlelocation": "left",
    "axes.titlepad": 10,
    "axes.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "grid.linestyle": "-",
    "text.color": INK,
    "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "font.size": 10,
})


def grid_on(ax, axis="x"):
    """Hairline grid on one axis only, behind the marks."""
    ax.set_axisbelow(True)
    ax.grid(axis=axis)

## 1. What the schemas actually contain

Two things here are worth more than a row count: a column the data description
promises that does not exist, and a column that exists in train but not in test.

In [3]:
shapes = pd.DataFrame(
    [("train.csv", *train.shape), ("test.csv", *test.shape),
     ("train_series.csv", *series.shape), ("test_series.csv", *test_series.shape),
     ("sample_submission.csv", *sample_sub.shape)],
    columns=["file", "rows", "columns"],
).set_index("file")
display(shapes.style.format(thousands=","))

In [4]:
# Which columns exist on each side of the train/test divide?
schema = pd.DataFrame(index=sorted(set(train.columns) | set(test.columns)))
schema["in_train"] = [c in train.columns for c in schema.index]
schema["in_test"] = [c in test.columns for c in schema.index]
schema["is_target"] = [c in LABELS for c in schema.index]
display(schema.loc[~schema["is_target"]].drop(columns="is_target"))

print(f"'Report'     in train: {'Report' in train.columns}   in test: {'Report' in test.columns}")
print(f"'PatientSex' in train: {'PatientSex' in train.columns}   "
      f"(the data description lists it)")

Two findings from that table.

**`Report` is train-only.** Every training study carries free-text radiology, and the
competition is described as multimodal, but the test schema has no report column. The
text is therefore a *labelling* signal for training, not an input your inference code
can consume. A fusion model with a text encoder has nothing to read at test time.

**`PatientSex` is absent** despite appearing in the data description. There is also no
`PatientID`, which leaves the grouping unit for a split unverified — section 3 comes
back to that with actual evidence.

In [5]:
labelled_mask = train[LABELS].notna().all(axis=1)
n_labelled = int(labelled_mask.sum())
labelled = train[labelled_mask].copy()

print(f"studies with all 12 labels : {n_labelled:>6,}  ({n_labelled / len(train):.1%})")
print(f"studies with report only   : {int((~labelled_mask).sum()):>6,}")
print(f"partially labelled studies : "
      f"{int((train[LABELS].notna().any(axis=1) & ~labelled_mask).sum()):>6,}")

Labels are all-or-nothing — no study carries some of the twelve. Prevalence and
co-occurrence over these 58 are covered well elsewhere, so here they are as plain
tables rather than another pair of charts.

In [6]:
prev = (
    labelled[LABELS]
    .agg(["sum", "mean"])
    .T.rename(columns={"sum": "positives", "mean": "prevalence"})
    .sort_values("positives", ascending=False)
)
prev["positives"] = prev["positives"].astype(int)
display(prev.style.format({"prevalence": "{:.1%}"}))

In [7]:
# Jaccard, P(both | either) — symmetric, so each pair appears once. Ranked instead
# of drawn as a 12x12 matrix, which other EDAs already show.
M = labelled[LABELS].values.astype(bool)
pairs = []
for i in range(len(LABELS)):
    for j in range(i + 1, len(LABELS)):
        both = int((M[:, i] & M[:, j]).sum())
        either = int((M[:, i] | M[:, j]).sum())
        pairs.append({"finding A": LABELS[i], "finding B": LABELS[j],
                      "both": both, "jaccard": both / either if either else 0.0})
top_pairs = pd.DataFrame(pairs).sort_values("jaccard", ascending=False).head(10)
display(top_pairs.style.format({"jaccard": "{:.2f}"}).hide(axis="index"))

## 2. The language mix, and why the usual shortcut gets it wrong

The reports are multilingual. The quick way to sort them is a cascade of substring
tests — check for `'the '` and call it English, check for `'la '` and call it French,
and so on. That approach is widely used and it fails hard here, for a reason worth
seeing: **`'la '` is as common in Spanish as in French, and whichever language is
tested first wins.**

Below, that cascade is run alongside a two-stage detector and the two are compared.

Stage one of the better version is Unicode script: if the letters are mostly Greek or
Cyrillic the *script* is settled with no ambiguity, though the language is not —
Cyrillic covers Bulgarian, Russian, Serbian and more, so those stay labelled as
scripts. Stage two scores the Latin-script remainder by stopword counts across eight
candidate languages, and requires the winner to beat the runner-up by a clear margin.
It is still a heuristic; `unknown` is what it declines to call, reported rather than
guessed away.

In [8]:
def detect_language_naive(text):
    """The substring cascade in common use. Order of the tests decides the answer."""
    t = text.lower()
    if any(w in t for w in ["the ", "and ", "of the", "is ", "are "]):
        return "English"
    if any(w in t for w in ["la ", "le ", "et ", "du ", "des "]):
        return "French"
    if any(w in t for w in ["der ", "die ", "das ", "und ", "mit "]):
        return "German"
    if any(w in t for w in [" el ", " la ", " en ", " de ", " que "]):
        return "Spanish"
    return "unknown"


# Named as scripts, not languages, because that is all this stage can honestly claim.
SCRIPTS = {
    "Greek script": (0x0370, 0x03FF),
    "Cyrillic script": (0x0400, 0x04FF),
    "Hebrew script": (0x0590, 0x05FF),
    "Arabic script": (0x0600, 0x06FF),
    "CJK": (0x4E00, 0x9FFF),
    "Hangul": (0xAC00, 0xD7AF),
}

STOPWORDS = {
    "English": "the of and with is no in to are there without within left right knee normal",
    "Spanish": "de la el en se con del no por una los las que sin rodilla menisco hallazgos",
    "Portuguese": "de da do com em no na sem os as uma que dos joelho achados sinais",
    "French": "de la le des en avec du les sans une aux genou articulaire aucune signal",
    "Italian": "di la il del con in non una della dei ginocchio destro sinistro nella",
    "German": "der die und mit im des ist ein kein von keine nachweis knie rechts links",
    "Dutch": "van de het en met geen een is in bij niet knie rechts links bevindingen",
    "Turkish": "ve bir ile mrg diz sag sol bulgular tetkik normaldir izlenmedi eklem kemik",
}
STOPWORDS = {k: set(v.split()) for k, v in STOPWORDS.items()}

WORD_RE = re.compile(r"[^\W\d_]+", re.UNICODE)


def detect_language(text):
    """Unicode script first, then stopword scoring on Latin script."""
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return "unknown"
    for name, (lo, hi) in SCRIPTS.items():
        if sum(lo <= ord(c) <= hi for c in letters) / len(letters) > 0.15:
            return name

    tokens = Counter(WORD_RE.findall(text.lower()))
    scores = sorted(
        ((sum(tokens[t] for t in stop), lang) for lang, stop in STOPWORDS.items()),
        reverse=True,
    )
    (best, lang), (second, _) = scores[0], scores[1]
    # Needs enough evidence, and a clear win over the runner-up: the Romance
    # languages share function words, so a narrow margin is not a decision.
    return lang if best >= 3 and best >= 1.3 * second else "unknown"


reports = train["Report"].dropna()
lang = reports.apply(detect_language)
lang_naive = reports.apply(detect_language_naive)

mix = lang.value_counts()
mix = mix.loc[[n for n in mix.index if n != "unknown"]
              + (["unknown"] if "unknown" in mix.index else [])]
display(pd.DataFrame({"reports": mix, "share": mix / len(reports)})
        .style.format({"reports": "{:,}", "share": "{:.1%}"}))

print("what the substring cascade returns instead:")
print(lang_naive.value_counts().to_string())

The cascade returns four buckets for a nine-bucket corpus, and one of its four —
Spanish — never fires at all, because every Spanish report trips the French test
first. The chart below breaks that down: for each language the two-stage detector
identifies, how often the cascade agrees, contradicts it, or gives up.

In [9]:
agree = pd.DataFrame({"lang": lang, "naive": lang_naive})
rows = []
for name, g in agree.groupby("lang"):
    rows.append({
        "lang": name,
        "agrees": int((g["naive"] == g["lang"]).sum()),
        "contradicts": int((g["naive"] != g["lang"]).sum() - (g["naive"] == "unknown").sum()),
        "gives up": int((g["naive"] == "unknown").sum()),
    })
cmp_df = pd.DataFrame(rows).set_index("lang")
cmp_df = cmp_df.loc[cmp_df.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(8, 4.2))
left = np.zeros(len(cmp_df))
for col, color in [("agrees", BLUE), ("contradicts", ORANGE), ("gives up", MUTED)]:
    ax.barh(cmp_df.index, cmp_df[col], left=left, height=0.6, color=color, label=col)
    left += cmp_df[col].values
grid_on(ax, "x")
ax.set_xlabel("reports")
ax.set_title("Substring cascade vs script + stopword detection")
# Below the axis: a three-entry legend collides with the title on the same line.
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3)
plt.tight_layout()
plt.show()

worst = cmp_df["contradicts"].idxmax()
print(f"most contradicted language: {worst} "
      f"({cmp_df.loc[worst, 'contradicts']:,} reports)")
print(f"total reports the cascade gets wrong or abandons: "
      f"{int(cmp_df['contradicts'].sum() + cmp_df['gives up'].sum()):,} of {len(reports):,}")

The practical consequence: a report-based label extractor routed by the cascade sends
Spanish and Turkish text down a French branch and never sees Greek or Cyrillic at all.
The failure is silent, and it correlates with imaging site rather than scattering at
random — so it will not average out.

## 3. Report text is not unique

Worth checking before treating one report as one exam, or using text similarity to
guess at the missing `PatientID`.

In [10]:
counts = reports.value_counts()
dups = counts[counts > 1]
affected = int(dups.sum())

print(f"distinct report texts        : {len(counts):,}")
print(f"texts used by >1 study       : {len(dups):,}")
print(f"studies sharing text         : {affected:,} of {len(train):,} "
      f"({affected / len(train):.1%})")
print(f"largest group                : {int(dups.max())} studies with identical text")
print(f"\nmedian words, duplicated texts : {dups.index.to_series().str.split().str.len().median():.0f}")
print(f"median words, all reports      : {reports.str.split().str.len().median():.0f}")

In [11]:
group_sizes = dups.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(group_sizes.index.astype(str), group_sizes.values, width=0.6, color=BLUE)
grid_on(ax, "y")
ax.set_xlabel("studies sharing one identical report text")
ax.set_ylabel("report texts")
ax.set_title("Duplicate report groups")
ax.set_ylim(0, group_sizes.max() * 1.15)
for x, y in zip(range(len(group_sizes)), group_sizes.values):
    ax.text(x, y + group_sizes.max() * 0.03, f"{y}", ha="center", fontsize=9, color=INK_SOFT)
plt.tight_layout()
plt.show()

In [12]:
top_dups = pd.DataFrame({
    "studies": dups.head(6).values,
    "words": [len(t.split()) for t in dups.head(6).index],
    "language": [detect_language(t) for t in dups.head(6).index],
    "text": [t.strip().replace("\n", " ")[:70] + "..." for t in dups.head(6).index],
})
display(top_dups.style.hide(axis="index"))

The duplicated texts are short — a median of 26 words against 129 across the corpus —
and they read as normal-study templates: *"Técnica: RMN de la rodilla. Resultados: Sin
anomalías."* One such template covers 37 studies.

Two consequences.

**Identical text does not mean the same patient.** These are boilerplate negatives, so
report text cannot substitute for the missing `PatientID`, and deduplicating on text
would throw away legitimately distinct studies.

**A random split leaks anyway.** 177 studies share text with another study, so a
random assignment puts byte-identical reports on both sides of the split. Any model
reading the report memorises the pair rather than generalising, and the validation
score for those studies is free. Grouping on report text before splitting costs
nothing and removes the effect.

## 4. `Fluid_Sensitive` and `Fat_Suppression` are the same column

`train_series.csv` offers two sequence flags. They describe genuinely different
physics — T2/PD/STIR weighting versus fat suppression — and real protocols routinely
have one without the other, so they should disagree on a large share of series.

In [13]:
differ = int((series["Fluid_Sensitive"] != series["Fat_Suppression"]).sum())
print(f"series where Fluid_Sensitive != Fat_Suppression: {differ:,} of {len(series):,}")
print(f"identical on every row: {differ == 0}")

t_differ = int((test_series["Fluid_Sensitive"] != test_series["Fat_Suppression"]).sum())
print(f"same check on test_series.csv: {t_differ:,} of {len(test_series):,} differ")

Zero disagreement on either side. This is almost certainly a defect in the released
files rather than a property of knee MRI — two columns carrying one column's worth of
information. Treat it as a single feature, expect a possible re-release, and do not
build a plane-and-sequence encoding that assumes the two are independent.

Since the flag is real information even if it is duplicated, here is the one view of
it that is not just a marginal count — how sequence type splits across the three
planes:

In [14]:
plane_by_flag = (
    series.groupby(["Anatomical_Plane", "Fluid_Sensitive"]).size().unstack(fill_value=0)
)
share = plane_by_flag.div(plane_by_flag.sum(axis=1), axis=0)
display(plane_by_flag.style.format("{:,}"))

cols = list(plane_by_flag.columns)
x = np.arange(len(plane_by_flag.index))
h = 0.36
fig, ax = plt.subplots(figsize=(8, 3.6))
for k, (col, color) in enumerate(zip(cols, [BLUE, ORANGE])):
    offset = (k - (len(cols) - 1) / 2) * (h + 0.04)
    bars = ax.bar(x + offset, plane_by_flag[col], width=h, color=color,
                  label=f"Fluid_Sensitive = {col}")
    for bar, frac in zip(bars, share[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 120,
                f"{frac:.0%}", ha="center", fontsize=8, color=INK_SOFT)
grid_on(ax, "y")
ax.set_xticks(x, plane_by_flag.index)
ax.set_ylabel("series")
ax.set_title("Sequence type by plane  ·  labels are the within-plane share")
ax.set_ylim(0, plane_by_flag.values.max() * 1.15)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
plt.tight_layout()
plt.show()

Axial series are 80% fluid-sensitive while sagittal are barely half. So the flag is
not independent of plane, and a per-plane model sees a different sequence mix in each
branch — relevant if you plan separate heads per plane.

## What this adds up to

1. **Reports label, they do not predict.** `Report` is train-only, so the text is how
   you get labels for the 4,349 unlabelled studies, and a text encoder has no input at
   inference. Plan the pipeline as extraction-then-imaging, not fusion.
2. **Extraction has to be genuinely multilingual.** Seven Latin-script languages plus
   Greek and Cyrillic, and the substring cascade that looks like it handles this
   silently routes two of the largest groups to the wrong branch.
3. **Group on report text before splitting.** 177 studies share text with another
   study. It is a one-line fix and it removes a real train/validation overlap.
4. **Treat the two sequence flags as one.** They are byte-identical in train and test.
5. **No `PatientID`, no `PatientSex`.** The grouping unit is unverified beyond the text
   duplicates above, so whether one patient contributes several studies is still open.

*Reproducible from the tabular files alone — no DICOM, no internet, a few seconds on
CPU. Corrections and disagreement welcome in the comments.*